# Fruit Segmentation — Google Colab Training

Trains **ConvNeXt-V2 U-Net** or **Swin-V2 U-Net** on the Fresh & Rotten Fruits dataset.

**Runtime**: Runtime → Change runtime type → **T4 GPU**

---
### Steps
1. Mount Google Drive
2. Clone repo & install dependencies
3. Upload / link dataset
4. Choose model config
5. Train
6. Visualise results

In [1]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ── 2. Clone repo and install ────────────────────────────────────────────
import os

REPO_URL = "https://github.com/jibran/fruit_segmentation.git"  # update this
REPO_DIR = "/content/fruit-segmentation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Install uv then sync dependencies
# !pip install uv -q
# !uv pip install -e . --system -q
!pip install timm -q  # for timm-based models
!pip install opencv-python-headless -q  # for image processing

print("✓ Dependencies installed")

Cloning into '/content/fruit-segmentation'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 55 (delta 10), reused 55 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 63.66 KiB | 15.92 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/fruit-segmentation
✓ Dependencies installed


In [3]:
# ── 3. Dataset setup ─────────────────────────────────────────────────────
# Option A: download from Kaggle
# !pip install kaggle -q
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
# !kaggle datasets download -d <dataset-slug> -p data/raw --unzip

# Option B: copy preprocessed data from Drive
DRIVE_DATA = "/content/drive/MyDrive/fruit_segmentation_data"
LOCAL_DATA = "/content/fruit-segmentation/data"
ZIP_DATA = f"{DRIVE_DATA}/processed.zip"


import os, shutil

# Extract the zip file to LOCAL_DATA if it exists
if os.path.exists(ZIP_DATA):
    shutil.unpack_archive(ZIP_DATA, LOCAL_DATA)
    print(f"✓ Dataset extracted to {LOCAL_DATA}")

if os.path.exists(DRIVE_DATA):
    if not os.path.exists(LOCAL_DATA):
        shutil.copytree(DRIVE_DATA, LOCAL_DATA)
    print(f"✓ Dataset at {LOCAL_DATA}")
else:
    print("⚠ Dataset not found — update DRIVE_DATA path or use Option A above")

✓ Dataset extracted to /content/fruit-segmentation/data
✓ Dataset at /content/fruit-segmentation/data


In [4]:
# ── 4. Choose model and size ─────────────────────────────────────────────
MODEL = "swin_unet"  # "convnext_unet" or "swin_unet"
SIZE = "tiny"  # "tiny", "small", or "base"

CONFIG = f"config/{MODEL}.yaml"
print(f"Config: {CONFIG}  |  Size: {SIZE}")

Config: config/swin_unet.yaml  |  Size: tiny


In [5]:
# ── 5. Verify GPU ────────────────────────────────────────────────────────
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [7]:
# ── 6. Quick model sanity check ──────────────────────────────────────────
import sys

sys.path.insert(0, "/content/fruit-segmentation")

from config.config_loader import load_config
from models import build_model

cfg = load_config(CONFIG)
cfg["model"]["size"] = SIZE

model = build_model(cfg)
counts = model.count_parameters()
print(f"Model    : {cfg['model']['name']}_{SIZE}")
print(f"Backbone : {counts['backbone']/1e6:.1f}M params")
print(f"Decoder  : {counts['decoder']/1e6:.1f}M params")
print(f"Total    : {counts['total']/1e6:.1f}M params")

# Forward pass smoke test
x = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    out = model(x)
print(f"Output shape: {out.shape}  ✓")

Model    : swin_unet_tiny
Backbone : 27.6M params
Decoder  : 5.3M params
Total    : 32.8M params
Output shape: torch.Size([1, 17, 512, 512])  ✓


In [ ]:
# ── 7. Train ─────────────────────────────────────────────────────────────
!python train/train.py \
    --config {CONFIG} \
    --size {SIZE} \
    --device cuda


Fruit Segmentation Training
  Model : swin_unet_tiny
  Device: cuda
  Config: config/swin_unet.yaml
  [sampler] Computing per-sample weights for WeightedRandomSampler...
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  Params: 32.8M total (27.6M backbone + 5.3M decoder)
  Logs  : logs/swin_unet_tiny-202603300854.csv

  Phase 1 — Backbone FROZEN
  Epochs: 15  |  Device: cuda
Epoch 1 [train]:   0% 0/310 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. 

In [ ]:
# Copy the logs folder to GDrive
os.makedirs("/content/drive/MyDrive/fruit_segmentation_logs", exist_ok=True)
shutil.copytree(
    "logs", "/content/drive/MyDrive/fruit_segmentation_logs", dirs_exist_ok=True
)

# Copy the checkpoint folder to GDrive
os.makedirs("/content/drive/MyDrive/fruit_segmentation_checkpoints", exist_ok=True)
shutil.copytree(
    "checkpoints",
    "/content/drive/MyDrive/fruit_segmentation_checkpoints",
    dirs_exist_ok=True,
)

In [ ]:
# ── 8. Visualise training curves ─────────────────────────────────────────
import glob
from visualization.plot_metrics import plot_training_curves, compare_logs_from_dir

# Find latest log for this model
pattern = f"logs/{MODEL}_{SIZE}-*.csv"
logs = sorted(glob.glob(pattern))
if logs:
    fig_path = plot_training_curves(logs[-1], show=True)
    print(f"Saved: {fig_path}")
else:
    print(f"No log files found matching {pattern}")

In [ ]:
# ── 9. Compare all trained models ────────────────────────────────────────
compare_logs_from_dir("logs/", metric="miou")

In [ ]:
# ── 10. Run inference on a sample image ──────────────────────────────────
import os

CKPT = f"checkpoints/best/{MODEL}_{SIZE}_best.pth"
SAMPLE_IMAGE = "data/processed/images/test/"  # directory or single file
OUTPUT_DIR = "predictions/"

if os.path.exists(CKPT):
    !python inference/inference.py \
        --config {CONFIG} \
        --checkpoint {CKPT} \
        --input {SAMPLE_IMAGE} \
        --output {OUTPUT_DIR} \
        --overlay
else:
    print(f"Checkpoint not found: {CKPT}")

In [ ]:
# ── 11. Display a prediction overlay ─────────────────────────────────────
import glob
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

overlays = sorted(Path(OUTPUT_DIR).glob("*_overlay.png"))
if overlays:
    fig, axes = plt.subplots(1, min(3, len(overlays)), figsize=(15, 5))
    if len(overlays) == 1:
        axes = [axes]
    for ax, path in zip(axes, overlays[:3]):
        ax.imshow(Image.open(path))
        ax.set_title(Path(path).stem, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No overlay images found — run the inference cell first.")

In [ ]:
# ── 12. Save results back to Drive ───────────────────────────────────────
import shutil, datetime

TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M")
DRIVE_SAVE = f"/content/drive/MyDrive/fruit_seg_results/{MODEL}_{SIZE}_{TIMESTAMP}"

os.makedirs(DRIVE_SAVE, exist_ok=True)

# Copy checkpoints and logs
for src in ["checkpoints", "logs"]:
    if os.path.exists(src):
        shutil.copytree(src, f"{DRIVE_SAVE}/{src}", dirs_exist_ok=True)

print(f"✓ Results saved to {DRIVE_SAVE}")